In [32]:
import pandas as pd
import re

trump_df = pd.read_csv('trump_cleaned_speeches.csv')
trump_identifiers = ["PRESIDENT TRUMP", "THE PRESIDENT", "President Donald J. Trump"]

In [29]:
import re

def extract_leader_speech(text, target_names):
    """
    Extracts speech for specific leaders by finding speaker identifiers 
    anywhere in the text, then cleans and joins their dialogue.
    """
    if not isinstance(text, str):
        return ""
    
    # Updated Regex:
    # 1. \b matches a word boundary (so it finds 'PRESIDENT' even after other text)
    # 2. ([A-Z][A-Z\s\.]+): captures the name (including dots for 'J.') before a colon
    speaker_pattern = r'\b([A-Z][A-Z\s\.]+):'
    
    # re.split with a capture group returns [pre-text, name, speech, name, speech...]
    parts = re.split(speaker_pattern, text)
    
    # If no speakers were found, parts will only have 1 element
    if len(parts) < 3:
        return ""
    
    leader_content = []
    
    # Metadata is at index 0. 
    # Names are at 1, 3, 5... 
    # Speech content is at 2, 4, 6...
    for i in range(1, len(parts), 2):
        speaker_name = parts[i].strip().upper()
        content = parts[i+1].strip()
        
        # Check if any of our target names are inside the found speaker name
        if any(target.upper() in speaker_name for target in target_names):
            # Remove parentheticals (Laughter), [Inaudible], etc.
            content = re.sub(r'\(.*?\)', '', content)
            content = re.sub(r'\[.*?\]', '', content)
            
            # Normalize whitespace
            content = " ".join(content.split())
            
            leader_content.append(content)
            
    return " ".join(leader_content)

In [31]:
def filter_journalist_questions(text):
    # This matches 'Q' followed by one or more whitespace characters
    # We wrap it in () to keep the 'Q' in the result list so we know where the split happened
    q_pattern = r'(\bQ\s+)'
    
    # Split the already-identified Trump text
    parts = re.split(q_pattern, text)
    
    clean_content = []
    
    # If no 'Q' was found, parts[0] is just the whole text
    if len(parts) == 1:
        return text

    # The first element (index 0) is usually the text BEFORE the first Q (Trump's speech)
    clean_content.append(parts[0].strip())
    
    # We iterate through the rest. 
    # If we find a 'Q' in the list, we know the NEXT element is a question, so we skip it.
    i = 1
    while i < len(parts):
        if parts[i].strip() == "Q":
            # Skip the 'Q' identifier AND the actual question that follows it
            i += 2 
        else:
            # This is the President's response/speech
            clean_content.append(parts[i].strip())
            i += 1
            
    # Join everything back together, removing the 'END' tag if it persisted
    final_text = " ".join(clean_content)
    return re.sub(r'\s*END\s*$', '', final_text).strip()

In [30]:
trump_df['Cleaned_data'] = trump_df['Content'].apply(
        lambda x: extract_leader_speech(x, trump_identifiers)
    )
trump_df.to_csv('trump_cleaned_speeches.csv', index=False, encoding='utf-8-sig')